# 🎙️ Lumini AI Studio - Free VoxCPM2 Voice Cloning Server (Google Colab GPU)
### 100% Free Zero-Shot Voice Cloning & 48kHz Studio Quality Speech Engine

---
### 📌 အသုံးပြုနည်းလမ်းညွှန်:
1. `Runtime -> Change runtime type -> T4 GPU` ရွေးထားကြောင်း စစ်ဆေးပါ။
2. အောက်ပါ Cell များကို **Play (Run)** နှိပ်ပါ။
3. ထွက်လာသော `https://xxxx.trycloudflare.com` URL ကို Lumini `.env` ထဲ ထည့်သွင်းပါ။

In [ ]:
# Step 1: Install Dependencies
!pip install -q voxcpm fastapi uvicorn soundfile torch torchaudio python-multipart pycloudflared

In [ ]:
# Step 2: Launch VoxCPM2 and Official Cloudflare Tunnel (Fresh No-Cache)
import os, subprocess, time, re

# 1. Clean up old processes & old tunnel cache
os.system("killall -9 cloudflared 2>/dev/null; fuser -k 8000/tcp 2>/dev/null; pkill -9 -f 'python app.py' 2>/dev/null; rm -rf /root/.cloudflared /tmp/tunnel.log")
time.sleep(2)

server_code = '''import os
import io
import sys
import uuid
import tempfile
from pathlib import Path
import soundfile as sf
import torch
import numpy as np
import re
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import Response, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
import uvicorn

app = FastAPI(title="Lumini VoxCPM2 Free GPU Engine")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

VOICE_PROMPTS_DIR = Path(tempfile.gettempdir()) / "lumini_voxcpm_prompts"
VOICE_PROMPTS_DIR.mkdir(parents=True, exist_ok=True)

print("⏳ Loading OpenBMB VoxCPM2 Model into GPU...")
from voxcpm import VoxCPM
model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
print("✅ VoxCPM2 Model loaded successfully into GPU!")

@app.get("/api/voxcpm/status")
@app.get("/health")
async def get_status():
    cuda_available = torch.cuda.is_available()
    device_name = torch.cuda.get_device_name(0) if cuda_available else "CPU"
    return {
        "online": True,
        "ok": True,
        "engine": "VoxCPM2 (OpenBMB 48kHz High-Fidelity)",
        "cuda": cuda_available,
        "device": device_name,
        "sample_rate": 48000,
        "supports_zero_shot": True
    }

@app.post("/api/voxcpm/clone")
async def register_voice_sample(
    name: str = Form(...),
    instruction: str = Form("Energetic movie recap narration style"),
    transcript: str = Form(None),
    file: UploadFile = File(...)
):
    voice_id = str(uuid.uuid4())
    ext = Path(file.filename or "sample.wav").suffix or ".wav"
    target_path = VOICE_PROMPTS_DIR / f"{voice_id}{ext}"
    
    content = await file.read()
    with open(target_path, "wb") as f:
        f.write(content)

    return {
        "voiceId": voice_id,
        "name": name,
        "instruction": instruction,
        "message": "Voice sample registered successfully."
    }

def split_burmese_text(text: str, max_chunk_len: int = 140) -> list[str]:
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    raw_sentences = re.split(r'(?<=[။!?\n])\s*', text)
    chunks = []
    current = ""
    for s in raw_sentences:
        s = s.strip()
        if not s:
            continue
        if len(current) + len(s) <= max_chunk_len:
            current = (current + " " + s).strip() if current else s
        else:
            if current:
                chunks.append(current)
            if len(s) > max_chunk_len:
                sub_parts = re.split(r'(?<=[၊,])\s*', s)
                sub_curr = ""
                for sp in sub_parts:
                    sp = sp.strip()
                    if not sp:
                        continue
                    if len(sub_curr) + len(sp) <= max_chunk_len:
                        sub_curr = (sub_curr + " " + sp).strip() if sub_curr else sp
                    else:
                        if sub_curr:
                            chunks.append(sub_curr)
                        sub_curr = sp
                if sub_curr:
                    current = sub_curr
                else:
                    current = ""
            else:
                current = s
    if current:
        chunks.append(current)
    return chunks if chunks else [text]

@app.post("/api/voxcpm/synthesize")
async def synthesize_speech(
    voice_id: str = Form(...),
    text: str = Form(...),
    instruction: str = Form(None),
    speed: float = Form(1.0)
):
    audio_path = None
    for ext in [".wav", ".mp3", ".webm", ".m4a", ".ogg"]:
        candidate = VOICE_PROMPTS_DIR / f"{voice_id}{ext}"
        if candidate.exists():
            audio_path = candidate
            break

    if not audio_path:
        raise HTTPException(status_code=404, detail="Voice sample not found.")

    try:
        chunks = split_burmese_text(text, max_chunk_len=130)
        print(f"🎙️ Synthesizing {len(chunks)} Burmese chunks with High-Fidelity Diffusion...")

        audio_pieces = []
        sample_rate = 48000
        if hasattr(model, 'tts_model') and hasattr(model.tts_model, 'sample_rate'):
            sample_rate = model.tts_model.sample_rate

        pause_samples = int(sample_rate * 0.08)
        pause_array = np.zeros(pause_samples, dtype=np.float32)

        for idx, chunk in enumerate(chunks):
            if not chunk.strip():
                continue

            if audio_path and os.path.exists(audio_path) and os.path.getsize(audio_path) > 100:
                wav_chunk = model.generate(
                    text=chunk,
                    reference_wav_path=str(audio_path),
                    cfg_value=2.5,
                    inference_timesteps=15
                )
            else:
                wav_chunk = model.generate(
                    text=chunk,
                    cfg_value=2.5,
                    inference_timesteps=15
                )

            if isinstance(wav_chunk, torch.Tensor):
                wav_chunk = wav_chunk.detach().cpu().numpy()
            
            if isinstance(wav_chunk, np.ndarray):
                wav_chunk = wav_chunk.squeeze()

            if hasattr(wav_chunk, 'dtype') and wav_chunk.dtype == np.float64:
                wav_chunk = wav_chunk.astype(np.float32)

            audio_pieces.append(wav_chunk)
            if idx < len(chunks) - 1:
                audio_pieces.append(pause_array)

        if not audio_pieces:
            raise HTTPException(status_code=400, detail="No audio chunks generated.")

        full_audio = np.concatenate(audio_pieces)

        max_val = np.max(np.abs(full_audio))
        if max_val > 0.01:
            full_audio = (full_audio / max_val) * 0.95

        buf = io.BytesIO()
        sf.write(buf, full_audio, sample_rate, format='WAV')
        buf.seek(0)
        return Response(content=buf.read(), media_type="audio/wav")
    except Exception as e:
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == '__main__':
    uvicorn.run(app, host='0.0.0.0', port=8000)
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(server_code)

# 2. Install official cloudflared binary
if not os.path.exists("/usr/local/bin/cloudflared") and not os.path.exists("/usr/bin/cloudflared"):
    os.system("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared")

# 3. Start FastAPI server
proc_app = subprocess.Popen(['python', 'app.py'])
time.sleep(6)

# 4. Start fresh official cloudflared tunnel
tunnel_log = open('/tmp/tunnel.log', 'w')
proc_tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--metrics', 'localhost:20241'], stdout=tunnel_log, stderr=tunnel_log)

# 5. Extract 100% fresh URL
fresh_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists('/tmp/tunnel.log'):
        with open('/tmp/tunnel.log', 'r') as f:
            content = f.read()
            matches = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if matches:
                fresh_url = matches[-1]
                break

print("\n" + "="*60)
print("🎉 VoxCPM2 Free GPU Server is LIVE and RUNNING!")
print(f"👉 Copy this 100% FRESH URL for Lumini .env:\n{fresh_url}")
print("="*60 + "\n")

proc_app.wait()

